# Exercícios Pandas parte 2-2

Utilize os datasets do arquivo bike.zip e o esquema abaixo para responder às perguntas a seguir:
<img src="https://api-club-file.cb.hotmart.com/public/v5/files/c59ddd48-8d71-4e7b-946f-8b75817277b9"/>

# Importando bibliotecas e carregando datasets

In [12]:
import pandas as pd
import numpy as np

brands = pd.read_csv("/content/brands.csv", sep=",")
products = pd.read_csv("/content/products.csv", sep=",")
order_item = pd.read_csv("/content/order_items.csv", sep=",")
order = pd.read_csv("/content/orders.csv", sep=",")
store = pd.read_csv("/content/stores.csv", sep=",")
customer = pd.read_csv("/content/customers.csv", sep=",")
stock = pd.read_csv('/content/stocks.csv', sep=',')
staff = pd.read_csv('/content/staffs.csv', sep=',')

In [15]:
# Célula usada para verificar
order_item.head(5)

,order_id,item_id,product_id,quantity,list_price,discount
0,1,1,20,1,599.99,0.20
1,1,2,8,2,1799.99,0.07
2,1,3,10,2,1549.00,0.05
3,1,4,16,2,599.99,0.05
4,1,5,4,1,2899.99,0.20


# Exercício 7

Você trabalha numa empresa de bicicletas e precisa fazer uma clusterização separando marcas caras de baratas. Para isso, você precisará saber a média de preços das bicicletas de cada marca.

Dica: No dataset brands, você encontra as marcas e no products você possui o preço de cada bicicleta. Traga a informação de preço de products para brands usando a coluna brand_id, e, em seguida, faça o agrupamento de preço médio por cada brand.

In [3]:
# Juntei as duas tabelas que serão usadas na resolução
media_preco = brands.merge(products, on="brand_id", how="left")

# "Filtrei" as duas colunas que interessam: marca e preço!
media_preco = media_preco.groupby('brand_name')['list_price'].mean().reset_index()

# Renomeei as colunas para melhor legibilidade
media_preco = media_preco.rename(columns={'brand_name': 'Marca', 'list_price': 'Média_Preço'})

# Arredondei os números para duas colunas decimais, para melhor leitura
round(media_preco, 2)

,Marca,Média_Preço
0,Electra,761.01
1,Haro,621.99
2,Heller,2173.00
3,Pure Cycles,442.33
4,Ritchey,749.99
5,Strider,209.99
6,Sun Bicycles,524.47
7,Surly,1331.75
8,Trek,2500.06


## Aprendizados:

- Aprendi a usar merge, além da sua diferença entre join.
- Consolidei o uso do método groupby.
- Aprendi a usar a função rename, usada para renomear linhas, indexes e, no caso do exercício, colunas.

# Exercício 8

Agora que você já possui o preço médio de cada marca, você deve separar os valores em 3 grupos, sendo que os nomes de cada agrupamento será a partir dessa lista: ["Barato", "Médio", "Caro"].

Dica: Será que cut ou qcut podem resolver este problema para você?

In [5]:
# Criei uma nova coluna na tabela do exercício anterior.
# Dividi em três quartis e classifiquei entre barato, médio e caro.
media_preco['Rótulo'] = pd.qcut(media_preco['Média_Preço'], q=3, labels=['Barato', 'Médio', 'Caro'])
media_preco

,Marca,Média_Preço,Rótulo
0,Electra,761.006186,Médio
1,Haro,621.990000,Médio
2,Heller,2172.996667,Caro
3,Pure Cycles,442.333333,Barato
4,Ritchey,749.990000,Médio
5,Strider,209.990000,Barato
6,Sun Bicycles,524.468261,Barato
7,Surly,1331.753600,Caro
8,Trek,2500.064074,Caro


## Aprendizados:

- Usar qcut.
- Conceito de quartis na prática.

# Exercício 9

Outra demanda chegou, dessa vez precisamos entender se o preço tem alguma relação com as vendas. Bicicletas mais caras são mais vendidas que as mais baratas? Qual a diferença nas vendas entre cada um dos grupos que levantamos na última pergunta?

Dica: Agora, você precisará partir do DataFrame brands, trazer informações de produtos de cada marca e depois ir na order_items para trazer a quantidade de cada produto vendido. O primeiro join será com a coluna brand_id que é comum ao dataFrame brands e ao products. Na sequência, você consegue ver qual coluna existe em products e em order_items para fazermos mais um join? E o último agrupamento, como será feito? A pergunta é quanto cada marca vende, ou seja, o último agrupamento só pode ser com a coluna br...

In [6]:
preco = brands.merge(products, on="brand_id", how="left")
qte_vendida = preco.merge(order_item, on="list_price", how="left")

qte_vendida = qte_vendida.groupby('brand_name').agg({
    'list_price': 'mean', # Apliquei média para coluna 'list_price'
    'quantity': 'sum' # Apliquei soma para a coluna 'quantity'
}).reset_index()

qte_vendida_ordenada = qte_vendida.sort_values(by='quantity', ascending=False)
round(qte_vendida_ordenada, 2)

,brand_name,list_price,quantity
0,Electra,658.43,15621.0
8,Trek,2311.23,10614.0
7,Surly,1041.87,3500.0
6,Sun Bicycles,516.41,1697.0
1,Haro,502.60,834.0
3,Pure Cycles,444.72,619.0
4,Ritchey,749.99,222.0
2,Heller,1469.88,147.0
5,Strider,230.50,57.0


## Aprendizados:

- Usar o .agg no groupby para aplicar funções em diferentes colunas.
- Usar o método ascending para colocar ascendente ou descendente.

# Exercício 10

Uma das coisas mais importantes para sua marca é expandir pelo maior número de estados possíveis. Para isso, precisamos entender quais estados consomem pouco e quais estados consomem muitos produtos. Em outras palavras, precisamos entender de onde vem a receita e onde precisamos gastar mais com marketing, ou entender se seria melhor simplesmente desligar a operação no local. Crie uma tabela sumarizando os gastos por estado, verifique qual a diferença do estado que mais consome para o estado que menos consome.

Sem dica para a última tarefa!

In [27]:
df = order.merge(order_item, on='order_id', how='left')
df1 = df.merge(customer, on='customer_id', how='left')
df1.head(5)

gasto_estado = df1.groupby('state').agg({
    'quantity': 'sum',
    'list_price': 'sum'})

round(gasto_estado, 2)

,quantity,list_price
state,,
CA,1516,1191373.67
NY,4779,3894954.72
TX,783,640078.18
